# 04 Travel times

This notebook builds the travel time matrix: how many minutes it takes to drive from each of the 3,465 origins to each of the 13 attractions. The model in 05 uses these times for distance decay (people visit far places less often).

**Snapping.** Origins and attractions are points with coordinates, but a route can only start and end at a network node (a road junction or road end). Snapping moves every point to its nearest node. The network has major roads only, so a village can be a few km from its nearest node; that gap is added as a short access time on local roads.

**Dijkstra's algorithm.** It finds the fastest routes from one start node to every other node in a single run, by always continuing from the closest node it has not finished yet. So one run per attraction (13 runs) gives all 3,465 × 13 times, instead of 45,045 separate route searches.

**Reversed graph.** Dijkstra measures times *from* its start node, but we need times from the origins *to* an attraction. Running it on a copy of the network where every road points the other way gives exactly those times, and one-way roads (ramps, divided highways) are still respected.

Times use the calibrated edge weight `travel_time` from 03 (weekend morning, slow zone around St Petersburg). Output: `data/processed/travel_times.csv`.

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

RAW = "../data/raw"
PROCESSED = "../data/processed"
CRS_M = "EPSG:32636"  # UTM zone 36N, metres

# travel_time is converted to numbers automatically; our own travel_time_ff column needs telling
G = ox.load_graphml(os.path.join(RAW, "roads_drive.graphml"), edge_dtypes={"travel_time_ff": float})
origins = pd.read_csv(os.path.join(PROCESSED, "origins.csv"))
att = pd.read_csv(os.path.join(PROCESSED, "attractions_geo.csv"))

tt = [d["travel_time"] for _, _, d in G.edges(data=True)]
print(f"Network: {len(G):,} nodes, {G.number_of_edges():,} edges, CRS {G.graph['crs']}")
print(f"travel_time is numeric: {isinstance(tt[0], float)}, mean {np.mean(tt):.1f} s per edge")
print(f"Origins: {len(origins):,}, population {origins['population'].sum():,.0f}")
print(f"Attractions: {len(att)}", att["role"].value_counts().to_dict())

Network: 9,789 nodes, 20,074 edges, CRS epsg:4326
travel_time is numeric: True, mean 114.0 s per edge
Origins: 3,465, population 8,265,799
Attractions: 13 {'calibration': 9, 'context': 3, 'competitor': 1}


In [2]:
# Snap every origin and attraction to its nearest network node
# (graph is in lat/lon, so osmnx uses scikit-learn and returns distances in metres)
origins["node"], origins["snap_m"] = ox.distance.nearest_nodes(
    G, X=origins["lon"], Y=origins["lat"], return_dist=True)
att["node"], att["snap_m"] = ox.distance.nearest_nodes(
    G, X=att["lon"], Y=att["lat"], return_dist=True)

# Snap distance summary for origins
km = origins["snap_m"] / 1000
print("Origin snap distance, km:")
print(km.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())
for limit in [2, 5, 10]:
    far = origins[km > limit]
    print(f"Origins > {limit} km from a node: {len(far):,} "
          f"({far['population'].sum() / origins['population'].sum():.1%} of population)")
print(f"Distinct nodes used by origins: {origins['node'].nunique():,}")

print("\n10 farthest origins:")
print(origins.nlargest(10, "snap_m")[["origin_id", "region", "population", "snap_m"]]
      .round(0).to_string(index=False))

print("\nAttractions:")
print(att[["name_en", "role", "node", "snap_m"]].round(0).to_string(index=False))

Origin snap distance, km:
count    3465.00
mean        5.73
std         6.20
min         0.02
50%         4.19
90%        12.64
99%        25.64
max        90.17
Origins > 2 km from a node: 2,430 (8.4% of population)
Origins > 5 km from a node: 1,495 (1.8% of population)
Origins > 10 km from a node: 580 (0.5% of population)
Distinct nodes used by origins: 1,385

10 farthest origins:
 origin_id           region  population  snap_m
      2665 Leningrad Oblast        12.0 90167.0
      2377 Leningrad Oblast        28.0 79025.0
      1997 Leningrad Oblast        65.0 78468.0
      1613 Leningrad Oblast       121.0 77556.0
      1339 Leningrad Oblast       186.0 76807.0
      3137 Leningrad Oblast         2.0 76337.0
      2032 Leningrad Oblast        61.0 59356.0
      3023 Leningrad Oblast         4.0 46526.0
      2309 Leningrad Oblast        35.0 46261.0
      3210 Leningrad Oblast         2.0 45860.0

Attractions:
                                  name_en        role       node  snap_m

## Access time and unreachable origins

The snapping check found 25 origins more than 30 km from the network (788 people): small islands in the Gulf of Finland and Lake Ladoga, and a roadless forest area in the far east of the oblast near the Vologda border. They cannot reach the attractions by car along this network and are dropped.

For all other points, the gap between the point and its node gets an access time: straight-line distance × 1.3 (real roads wind) at 40 km/h (local roads). It is added at both ends of every trip, origin side and attraction side. Four attractions sit 2-10 km from the major network (Izvara, Staraya Ladoga, Putilovo, Gdov), so their times are checked against Yandex below.

In [4]:
MAX_SNAP_KM = 30   # farther than this = island or roadless forest, not reachable by car
ACCESS_KMH = 40    # average speed on local roads between a point and the major network
DETOUR = 1.3       # real roads are longer than the straight line

dropped = origins[origins["snap_m"] > MAX_SNAP_KM * 1000]
origins = origins[origins["snap_m"] <= MAX_SNAP_KM * 1000].reset_index(drop=True)
print(f"Dropped {len(dropped)} origins ({dropped['population'].sum():,.0f} people), "
      f"{len(origins):,} origins left")

def access_min(snap_m):
    return snap_m / 1000 * DETOUR / ACCESS_KMH * 60

origins["access_min"] = access_min(origins["snap_m"])
att["access_min"] = access_min(att["snap_m"])

print("\nOrigin access time, min:")
print(origins["access_min"].describe(percentiles=[0.5, 0.9]).round(1).to_string())
print("\nAttraction access time:")
print(att[["name_en", "snap_m", "access_min"]].round(1).to_string(index=False))

Dropped 25 origins (788 people), 3,440 origins left

Origin access time, min:
count    3440.0
mean       10.6
std         9.5
min         0.0
50%         8.1
90%        24.2
max        57.9

Attraction access time:
                                  name_en  snap_m  access_min
             Slantsy Local History Museum   819.0         1.6
           Kingisepp Local History Museum   321.7         0.6
Ivangorod Museum (fortress and galleries)   452.6         0.9
                         Koporye Fortress   619.8         1.2
            Staraya Ladoga Museum-Reserve  4555.0         8.9
             Sablino caves and waterfalls   354.4         0.7
      Putilovo Paleopark (fossil digging)  2419.9         4.7
            Gdov Kremlin and Local Museum  2082.1         4.1
          Oreshek Fortress (Shlisselburg)   760.1         1.5
           Roerich Estate Museum (Izvara)  9616.3        18.8
        Yam Fortress ramparts (Kingisepp)    95.2         0.2
                 Lopukhinka radon springs

In [5]:
# Reversed graph: every road points the other way, so Dijkstra FROM an attraction
# gives the fastest times TO that attraction from every node (one-way roads respected)
G_rev = G.reverse(copy=False)

times = {}
for _, a in att.iterrows():
    sec = nx.single_source_dijkstra_path_length(G_rev, a["node"], weight="travel_time")
    network_min = origins["node"].map(sec) / 60
    times[a["name_en"]] = network_min + origins["access_min"] + a["access_min"]

tt = pd.DataFrame(times)
tt.index = origins["origin_id"]

print(f"Matrix: {tt.shape[0]:,} origins × {tt.shape[1]} attractions, missing values: {tt.isna().sum().sum()}")
print("\nTravel time per attraction, minutes:")
print(tt.describe().T[["min", "50%", "max"]].round(0).to_string())

Matrix: 3,440 origins × 13 attractions, missing values: 0

Travel time per attraction, minutes:
                                            min    50%    max
Slantsy Local History Museum                3.0  252.0  515.0
Kingisepp Local History Museum              3.0  216.0  470.0
Ivangorod Museum (fortress and galleries)   1.0  232.0  486.0
Koporye Fortress                            2.0  200.0  450.0
Staraya Ladoga Museum-Reserve              10.0  192.0  385.0
Sablino caves and waterfalls               11.0  185.0  368.0
Putilovo Paleopark (fossil digging)        10.0  180.0  358.0
Gdov Kremlin and Local Museum               9.0  291.0  558.0
Oreshek Fortress (Shlisselburg)             6.0  180.0  385.0
Roerich Estate Museum (Izvara)             20.0  199.0  447.0
Yam Fortress ramparts (Kingisepp)           2.0  216.0  469.0
Lopukhinka radon springs                    1.0  185.0  428.0
Lava river canyon                           7.0  182.0  352.0
